# Google Colab — CNN6 de 6 clases (CirCor + SPRSound)

Fine-tuning de la CNN6 preentrenada de PANNs como modelo único de 6 clases
(cardíaco normal / murmullo, pulmonar normal / crackle / wheeze / ambos), con la
receta de `cs_wide` y la red de la variante B de `avances/17`. Es el paso 2 de
`avances/25` §8: el techo de desempeño con la mejor receta.

**Qué hace cada cosa:**
- **Código**: se clona del repositorio de GitHub (la última versión subida).
- **Datos**: un único archivo `colab_cnn6_circor_spr.zip` (~1.15 GB) en tu Drive:
  el audio a 4 kHz de CirCor y SPRSound, los índices de ventanas, los folds y los
  pesos de CNN6. Se copia una sola vez al disco local de la VM y el audio entero
  se carga en la GPU; el espectrograma se calcula en la GPU.
- **Resultados**: se guardan en tu Drive, época por época.

**Antes de correr:**
1. Subir el código nuevo a GitHub (`git push`): este notebook usa
   `desarrollo/src/unified_model/run_cnn6_gpu.py`.
2. Subir `desarrollo/data/colab_cnn6_circor_spr.zip` (lo genera
   `unified_model/pack_cnn6_colab.py`) a la carpeta de Drive de la configuración.
3. Activar la GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.

**Si la sesión se corta**: volver a correr todas las celdas. Retoma desde la
última época guardada en Drive y no repite los folds terminados.

In [ ]:
print("=== Montando Google Drive ===")
from google.colab import drive
drive.mount('/content/drive')

## Configuración

**Plan** (el mismo protocolo de `avances/25`):

1. **Filtro**: los dos frontends en los folds 0 y 3, `FOLDS = "0 3"`, primero
   `FRONTEND = "panns32k"` y después `"ours64"`.
   - `panns32k`: el de PANNs, el que ganó en `avances/15`. ~30 bandas con contenido
     y pocas por debajo de 400 Hz.
   - `ours64`: 64 bandas de 50 a 2000 Hz. Más resolución para el murmullo, pero no
     coincide con lo que vio la red en el preentrenamiento.
2. Localmente, `filter_variants.py` compara con `cs_wide` solo en validación.
3. **5 folds** solo con el ganador: `FOLDS = "0 1 2 3 4"` con el mismo
   `FRONTEND`. Los folds 0 y 3 ya terminados se saltean.

Cada frontend guarda en su carpeta (`cnn6_panns32k`, `cnn6_ours64`) dentro de `OUT_DIR`.

In [ ]:
from pathlib import Path

# Editá estas rutas si usás otra carpeta de Drive.
DRIVE_DIR   = Path("/content/drive/MyDrive/CEIA/colab_cnn6")
PACKAGE_ZIP = DRIVE_DIR / "colab_cnn6_circor_spr.zip"
OUT_DIR     = DRIVE_DIR / "resultados"

FRONTEND = "panns32k"   # "panns32k" u "ours64"
FOLDS    = "0 3"        # filtro: "0 3"; 5 folds: "0 1 2 3 4"

REPO_URL = "https://github.com/SantiagoBettig/deteccion_patologias_cardiopulmonares.git"
REPO_DIR = Path("/content/repo")
PKG_DIR  = Path("/content/pkg")

assert PACKAGE_ZIP.exists(), f"No se encontró el paquete de datos: {PACKAGE_ZIP}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Paquete: {PACKAGE_ZIP} ({PACKAGE_ZIP.stat().st_size / 1e6:.0f} MB)")
print(f"✓ Resultados en: {OUT_DIR / ('cnn6_' + FRONTEND)}")
print(f"✓ Frontend: {FRONTEND} | folds: {FOLDS}")

In [ ]:
print("=== Verificando GPU ===")
import torch
print("CUDA disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución"
print("Dispositivo:", torch.cuda.get_device_name(0))

In [ ]:
print("=== Clonando / actualizando el repositorio ===")
import subprocess
if (REPO_DIR / ".git").exists():
    print(subprocess.run(["git", "-C", str(REPO_DIR), "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "desarrollo" / "src"
assert (SRC_DIR / "unified_model" / "run_cnn6_gpu.py").exists(),     "El repositorio no tiene run_cnn6_gpu.py: ¿hiciste git push del código nuevo?"
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
print("=== Dependencias ===")
# torch, numpy, pandas, scikit-learn, matplotlib, scipy y librosa ya vienen en Colab.
# No se instala requirements.txt para no arriesgar reemplazar el torch con GPU de Colab.
import librosa, sklearn, pandas, numpy, scipy
print("✓ librosa", librosa.__version__, "| torch", torch.__version__)

In [ ]:
print("=== Copiando el paquete de datos al disco local de la VM ===")
import json, shutil, time, zipfile
if not (PKG_DIR / "audio_4k_int16.npy").exists():
    t0 = time.time()
    local_zip = Path("/content/package.zip")
    shutil.copy2(PACKAGE_ZIP, local_zip)
    zipfile.ZipFile(local_zip).extractall(PKG_DIR)
    local_zip.unlink()
    print(f"✓ Copiado y descomprimido en {time.time() - t0:.0f} s")
print(sorted(p.name for p in PKG_DIR.iterdir()))
print(json.dumps(json.loads((PKG_DIR / "manifest.json").read_text()), indent=1))

## Entrenamiento

Una línea por época: loss de train / val, AUC de val (murmullo, crackle, wheeze),
accuracy de dominio, criterios A y B, segundos. `*` = mejoró el criterio principal
(B), `+` = mejoró el otro (A). Al terminar cada fold, el resultado de test por
ventana. Si la sesión se corta, volver a correr desde arriba: retoma solo.

In [ ]:
!cd {SRC_DIR} && python -u unified_model/run_cnn6_gpu.py     --package-dir {PKG_DIR} --out-dir {OUT_DIR} --frontend {FRONTEND} --folds {FOLDS}

## Después de entrenar

Los resultados quedan en `OUT_DIR/cnn6_<frontend>/` en tu Drive. Bajar esa carpeta
(entera, con los `.pt` si querés guardar los checkpoints) y, desde `desarrollo/src`:

```
python unified_model/import_colab_run.py --src <carpeta bajada>/cnn6_panns32k
python unified_model/import_colab_run.py --src <carpeta bajada>/cnn6_ours64
python unified_model/filter_variants.py --runs cs_wide cnn6_panns32k cnn6_ours64
```

Con los 5 folds del ganador:

```
python unified_model/import_colab_run.py --src <carpeta bajada>/cnn6_<ganador>
python unified_model/evaluate_circor_spr.py --run cnn6_<ganador> --base-run cs_wide
python unified_model/plot_circor_spr.py --run cnn6_<ganador> --base-run cs_wide
```